# OpenPlaque — Secondary Branch 3-D Vesselness / Topology v1.1

Bugfix rerun of the 3-D vesselness/topology experiment.

The scientific method and thresholds are unchanged. This version fixes the Dijkstra flat-index → `(z,y,x)` conversion that caused the Cell 9 `IndexError`.

The known ~10.6→12.7 mm segment remains a mandatory positive control before any distal inference is accepted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse expensive valid products; recompute all outputs affected by the indexing fix.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_VESSELNESS = True
REUSE_TOPOLOGY_SEARCH = False
REUSE_FIGURES = False
REUSE_REPORT = False


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil
if os.path.exists('/content/OpenPlaque'):
    shutil.rmtree('/content/OpenPlaque')
!git clone -q --depth 1 --branch secondary-3d-vesselness-topology-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%cd /content/OpenPlaque


In [ ]:
import sys
sys.path.insert(0, '/content/OpenPlaque/src')

from openplaque.secondary_3d_vesselness_topology import synthetic_vesselness_self_test
from openplaque.secondary_3d_vesselness_topology_v2 import (
    Secondary3DVesselnessTopologyWorkflow,
    finite_dist_coords,
)

self_test = synthetic_vesselness_self_test()
print('Synthetic 3-D vesselness self-test:', self_test)
assert self_test['passed'], 'Synthetic vesselness self-test failed'

# Regression test for the exact Cell 9 bug.
import numpy as np
_shape = (63, 50, 60)
_dist = np.full(np.prod(_shape), np.inf, dtype=np.float32)
_expected = [(0, 0, 0), (10, 20, 30), (62, 49, 59)]
for _p in _expected:
    _dist[np.ravel_multi_index(_p, _shape)] = 1.0
_got = [tuple(x) for x in finite_dist_coords(_dist, _shape).tolist()]
print('Flat-index regression test:', _got)
assert _got == _expected


In [ ]:
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'frozen_geometry': REUSE_FROZEN_GEOMETRY,
    'vesselness': REUSE_VESSELNESS,
    'topology_search': REUSE_TOPOLOGY_SEARCH,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = Secondary3DVesselnessTopologyWorkflow(
    root='/content/drive/MyDrive/OpenPlaque',
    reuse=reuse,
)
display(wf.cache_status())


In [ ]:
wf.load_source_ct()
geometry = wf.load_frozen_geometry(start_arc_mm=10.6, control_arc_mm=12.7)
display(geometry)


In [ ]:
vmeta = wf.compute_vesselness(scales_mm=(0.55, 0.80, 1.10, 1.45))
display(vmeta)


In [ ]:
summary = wf.search_topology(min_candidate_projection_mm=1.5, max_cost=150.0)
display(summary)
display(wf.candidates.head(12) if wf.candidates is not None and len(wf.candidates) else wf.candidates)


In [ ]:
figures = wf.make_figures()
print('Figures:', figures)
report = wf.make_report()
zip_path = wf.package()

print('Report:', report)
print('Final ZIP:', zip_path)
print()
print('Drive search link:')
print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_SECONDARY_3D_VESSELNESS_TOPOLOGY_REPORT_BACK.zip')
